## Tutorial setup

Run this cell **first**. It does two things:

1. Defines the GROMACS module line used everywhere (`$MOD`), so you load it the same way in every step.
2. Builds **short** versions of the NVT / NPT / production `.mdp` files so the whole workflow runs in a few minutes on the GPU instead of hours. These short runs are for *demonstration only* — real science needs the full-length runs in `../inp_files/`.

Run lengths chosen for the tutorial (Martini, dt = 20 fs):

| Step | steps | time |
| ---- | ----- | ---- |
| NVT  | 25 000  | 0.5 ns |
| NPT  | 50 000  | 1 ns |
| Prod | 250 000 | 5 ns |


In [ ]:
%%bash
# ===== Tutorial setup =====
# 1) Create a tidy directory structure for the workflow stages.
# 2) Build SHORT .mdp files for a fast demo run (instead of the full-length ones).
# Everything is run from the exercise/ directory; each stage writes into its own
# sub-folder. The force field is in ../toppar and the .mdp templates in ../inp_files.
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP

mkdir -p build em eq prod analysis

# Bring the starting all-atom structure into build/ (edit the name if different).
[ -f build/aa.pdb ] || cp aa.pdb build/ 2>/dev/null || true

# Short demo .mdp files, written into each stage folder.
INP=../inp_files
sed 's/^nsteps.*/nsteps = 25000/'  $INP/CG_nvt.mdp  > eq/CG_nvt_tut.mdp     # NVT 0.5 ns
sed 's/^nsteps.*/nsteps = 50000/'  $INP/CG_npt.mdp  > eq/CG_npt_tut.mdp     # NPT 1 ns
sed 's/^nsteps.*/nsteps = 250000/' $INP/CG_prod.mdp > prod/CG_prod_tut.mdp  # prod 5 ns

echo 'Folders:'; ls -d build em eq prod analysis
echo 'Short mdp files:'
grep -H '^nsteps' eq/CG_nvt_tut.mdp eq/CG_npt_tut.mdp prod/CG_prod_tut.mdp

# System preparation before preparing the glycoprotein

## Converting all atom to coarse grain

To start, we will take the all-atom structure we worked with in the previous tutorial. To convert it into a coarse-grained (CG) model, we will use a popular tool called `martinize2`. If you can somehow copy the CG files from the previous tutorial it would be grea, if not please follow along this part

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd build
# toppar and inp_files are one level above exercise/, i.e. two levels above build/
martinize2 -f aa.pdb -x CG.pdb -o topol.top -dssp -p backbone -elastic

## Preparing the glycoprotein system in coarse-grained level

Our main goal is to create a similar embedded system, but **we will instead modify the protein with glycosylation first before getting embedded into the membrane**. Here, we will employ the simple Python package called `Glycomarinate`, which is currently being developed in our group to prepare glycoprotein simulations using the Martini 3 force field.

There are two main commands in `Glycomarinate`:
1. `marinate_pdb`: generate the initial CG structure of a glycoprotein given the "vanilla" protein structure (both atomistic and martinize2 structures) and the desired glycosylation sites
2. `marinate_top`: generate the topology of CG structure of a glycoprotein using a graph-based pipeline to connect the topology of individual glycans into the side chains of the targeted amino acids in the protein

First of all, create a text file in `build` repository to store information about the target N-glycosylation sites along with the desired glycan types. Let's name the file as `glyco.txt`.

The file should contain the **protein chain, glycosylation site**, and **glycan type** in the following format:

   ```text
   Chain Glycosylation_Site Glycan_Name
   ```

For example:

   ```text
   A 304 Glycan_Name
   A 308 Glycan_Name
   ```

In this example, residues `304` and `308` are the glycosylation sites of our protein chain `A`

`Glycan_Name` must be chosen from the available glycan types in the `data/glycans` directory. Please use the exact directory name as the glycan name.

For the sake of simplicity, please use **MAN5** for both glycosylation sites and save the text file in the working directory

   ```text
   A 304 MAN5
   A 308 MAN5
   ```

## Grafting glycans into the initial coarse-grained structure of the protein 

We will modify the previous protein with two **MAN5** glycans as indicated by the `glyco.txt`, generating an output PDB file called `glyco_CG.pdb` using the following command:

In [ ]:
%%bash
marinate_pdb -i aa.pdb -c CG.pdb -g ../data/glycans/ -l glyco.txt -s 50 -t 0.7 -o glyco_CG.pdb

> **What happens here:** `marinate_pdb` utilizes GlycoSHIELD to select one of the non-clash glycan conformers for each glycosites, employing threshold distance criteria of 0.7 nm, which is default value in `GlycoSHIELD`

| Option         | Description                                                                                                                                                                 |
| -------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `-i aa.pdb`    | A PDB file of an atomistic protein. This is exactly the PDB used for the input in `martinize2`                                                                                                             |
| `-c CG.pdb`    | A PDB file of a CG structure generated from `martinize2`.                                                                                                                |
| `-g ../../data/glycans` | A directory of glycan library. In our case, it is located within the installation directory `/data/glycans` (check to ensure the location of this)                                                                               |
| `-l glyco.txt`        |  A txt file listing all glycosylation sites
| `-s 50`  | Number of skipped frames, a parameter in GlycoSHIELD. Default value is 50. For a crowded glycosylated environment such as spike proteins, smaller values might be required                                                            |
| `-t 0.7`     | Threshold distance, a parameter in GlycoSHIELD to filter glycan conformers based on given steric clash distances. Recommended to use the default value: 0.7 nm. For a crowded glycosylation environment, choose smaller values |
| `-o glyco_CG.pdb`     | The resulting output in PDB format |

Note: `Glycomarinate` is ideally installed with the default glycan library repository called `data`. If you don't find the `data` directory in you can use extract the `data.zip`:


In [ ]:
%%bash
unzip ../data.zip

## Build the topology of the glycoprotein

Next, we also modify the topology of the protein (`.itp` file) with two **MAN9** glycans as indicated by the `glyco.txt`, generating an output PDB file called `glyco_CG.itp` using the following command:

In [ ]:
%%bash
cd build
marinate_top -p aa.pdb -i molecule_0.itp -l glyco.txt -g ../data/glycans -r ../data/linkages/ASN_GYN.ff -o glyco_CG.itp

> **What happens here:** `marinate_itp` utilizes the `Polyply` package to represent the whole molecule as a graph with nodes and edges representing the beads and direct connection between beads, respectively, which is constructed from the `.itp` file directly. As the individual glycans have their own topology stored in `data/glycans` repository, `marinate_top` attempts to connect the glycan graphs to the protein graph in the selected glycosites. The link between the two graphs is specified by the so-called linkage file, which in our case is named as `ASN_GYN.ff`, describing the exact N-glycosylation linker parameters between the side chain bead of the asparagine residue and the beads of the first N-acetylglucosamine in the glycans

| Option         | Description                                                                                                                                                                 |
| -------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `-p aa.pdb`    | A PDB file of an atomistic protein. This is exactly the PDB used for the input in `martinize2`                                                                                                             |
| `-i molecule_0`    | A PDB file of a CG structure generated from `martinize2`.                                                                                                                |
| `-g ../../data/glycans` | A directory of glycan library. In our case, it is located within the installation directory `/data/glycans`                                                                                |
| `-l glyco.txt`        |  A txt file listing all glycosylation sites
| `-r ../../data/linkages/ASN_GYN.ff  | The linker parameter between the side chain and the glycan. In our case, the N-glycosylated linker (ASN-GYN) can be found in data/linkages                                                             |
| `-o glyco_CG.itp`     | The resulting topology output in ITP format |

## Modifying topology files to suit best for our simulations

Once again, we need to modify `topol.top` as the file generated is a template and missing some information about the other forcefield files. Open the `topol.top` file and add the following lines:

```
#include "../../toppar/martini_v3.0.0.itp"
#include "../../toppar/martini_v3.0.0_ffbonded_v2_openbeta.itp"
#include "../../toppar/martini_v3.0.0_phospholipids_PC_v2_openbeta.itp"
#include "../../toppar/martini_v3.0.0_solvents_v1.itp"
#include "../../toppar/martini_v3.0.0_ions_v1.itp"
#include "glyco_CG.itp"
```

In addition, due to a bug in `marinate_top` script, we also need to add the following lines before the header `[ moleculetype ]` in the `glyco_CG.itp`:

```
#ifndef POSRES_FC
#define POSRES_FC 1000
#endif
```

This is the default position restraint constants that exist in `molecule_0.itp` due to the command `-p backbone` from `martinize2`

We have successfully generated **two key input files** for running a CG simulation of the glycoprotein: `glyco_CG.itp` and `glyco_CG.pdb`. However, we will also proceed with **inserting this modified protein into a membrane system**. The rest of the tutorial will be **more or less similar to the previous one**. Therefore, we will copy the remaining stages from the previous notebook, with some minor changes.

## Adding a membrane to our system

Next step we will again use `insane` (**IN**sert membr**ANE**) to create a lipid bilayer of our choice (we will use POPC in this tutorial) and embed the glycoprotein. 

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd build
insane -f glyco_CG.pdb -p temp.top -o memb.gro -l POPC -u POPC -box 10,10,17

We need to copy the information about the number of lipid molecules from the temporary topology file, `temp.top`, to the previous `topol.top` for the next steps. For example, the last two lines of `temp.top` show the lipid type in the upper and lower leaflet and the corresponding number of lipids. Copy these two lines into `topol.top`:

```
POPC           164
POPC           165
```

NOTE: Make sure that the order of the molecules in the `[molecules]` section is the same as in the structure file (`.gro` or `.pdb`).

Now we have to add water and ions to the simulation box to make the system behave more like real biological or chemical environments. In MARTINI, one water bead is equivalent to four all-atom water molecules. To add water, we use the `solvate` command and a structure file of water provided as `water.gro`.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd build
gmx solvate -cp memb.gro -cs ../../inp_files/water.gro -o solvated.gro -radius 0.21 -p topol.top

**Next we will add ions.** This is to neutralise the total charge of the system and reproduce a physiological ion environment. This is a two-step process. The first step creates a binary input file (`.tpr`) needed for ion replacement. The second step uses `genion` to replace water beads with sodium (`NA`) and chloride (`CL`) ions at a given concentration (`-conc 0.15`). This will generate a coordinate file with ions (`ionized.gro`) and an updated `topol.top` with the number of ions added.


In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd build
gmx grompp -f ../../inp_files/CG_minim.mdp -c solvated.gro -p topol.top -o ionize -maxwarn 1
echo W | gmx genion -s ionize.tpr -o ionized -p topol.top -pname NA -nname CL -neutral -conc 0.15

# Molecular Dynamics Simulation

After the system is prepared, we can now proceed to the simulation


## Minimisation

> **What happens here:** building a system can leave beads slightly overlapping. Energy minimisation slides everything 'downhill' to the nearest low-energy arrangement, removing those clashes so dynamics can start safely.

The first step is to ensure that the energy of our system is minimized, or close to the minimum energy state. When we build the system, there is a chance that some atoms are placed too close together, which can cause steric clashes and unrealistic high energies.

The minimisation step moves atoms **downhill on the potential energy surface** to find a nearby low-energy, stable configuration. This step is used to remove *bad atomic contacts* and put the system into a physically reasonable starting shape before running dynamics.

The first step, `grompp`, generates a binary input file (`.tpr`) required for running energy minimisation. The second step, `mdrun`, performs the actual energy minimisation simulation.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd em
# inputs come from ../build ; mdp template is ../../inp_files
gmx grompp -f ../../inp_files/CG_minim.mdp -c ../build/ionized.gro -p ../build/topol.top -o emsol.tpr -maxwarn 1
gmx mdrun -deffnm emsol

# Equilibration

The next step of the simulation is equilibration. This is to bring the system into a **stable, physically realistic state** before collecting meaningful data in the production run. In this process, we bring the system to the target temperature and pressure in a step-by-step manner and prepare it for the final production run.

### Create index file

Before that, we need to create an *index file*, 

In the equilibration steps, we need to define groups for separate thermostat and barostat. In our system, we created four index groups: **glycoprotein** (protein and glycan/GYC), membrane, and solvent (water + ions).

The group **numbers** below (1 = Protein, 13 = GYC, 14 = POPC, 15 = W, 16=ION) are what `make_ndx` prints for *this* system. 

In [ ]:
%%bash
# Build index.ndx in em/ (groups: SOL, MEM, PROT) with GROMACS make_ndx.
# Group numbers (1=Protein, 13=POPC, 14=W, 15=ION) come from make_ndx for THIS system.
# If you change protein/membrane, run 'gmx make_ndx -f emsol.tpr', type only 'q',
# and read the numbers it prints.
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd em
gmx make_ndx -f emsol.tpr -o index.ndx << 'EOF'
1 | 13
15 | 16
name 17 GLYCOPROT
name 18 SOL
name 14 MEM
q
EOF

## NVT Equilibration

> **What happens here:** the thermostat gently scales particle velocities until the system reaches the target temperature, at fixed volume. Position restraints keep the protein roughly in place while the solvent settles.

**NVT equilibration** is used in molecular dynamics to stabilize the system at a **fixed number of particles (N), volume (V), and temperature (T)** before moving to pressure equilibration (NPT) and production runs.

After minimisation, the system is still not physically stable. NVT equilibration focuses on bringing the temperature to the target value in a controlled way.

This is done using a thermostat, which slowly adjusts the kinetic energy of the system to reach the desired temperature.

Like every simulation step, it has two stages. First, we generate a `.tpr` file for the next `mdrun` step.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd eq
# topology lives in ../build ; minimised structure + index in ../em ; mdp here in eq/
gmx grompp -f CG_nvt_tut.mdp -c ../em/emsol.gro -r ../em/emsol.gro \
    -p ../build/topol.top -n ../em/index.ndx -o nvt.tpr -maxwarn 1
gmx mdrun -deffnm nvt -nb gpu -bonded gpu -v

Note that in the `grompp` step, we used an extra parameter apart from `-n index.ndx` — `-r emsol.gro`. This is important for **position restraints** for the reference structure. It helps the structure not move too much during the equilibration steps compared to the solvent.

---

## NPT Equilibration

> **What happens here:** now the barostat is switched on. The box size is allowed to change so the system reaches the correct pressure and density. For a membrane the box relaxes mainly in the bilayer plane.

**NPT equilibration** is used in molecular dynamics to bring the system to the correct **pressure and density**, after temperature has already been stabilized in NVT.

In NPT, the system is kept at constant number of particles (N), pressure (P), and temperature (T).

This step adjusts the system density, distributes water molecules more evenly, and allows the box size to change to remove vacuum gaps or overly compressed regions.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd eq
# -t nvt.cpt carries velocities from NVT (both runs are in eq/)
gmx grompp -f CG_npt_tut.mdp -c nvt.gro -r nvt.gro \
    -p ../build/topol.top -t nvt.cpt -n ../em/index.ndx -o npt.tpr -maxwarn 1
gmx mdrun -deffnm npt -nb gpu -bonded gpu -v

Here, we added another parameter, `-t nvt.cpt`, which tells the program to use the velocities of the particles from the last simulation. This allows the simulation to continue smoothly from NVT and prevents the generation of random velocities for the particles.

---

## Production run

This is the final stage of the simulation workflow. After energy minimization and equilibration (NVT + NPT), the system is now ready for meaningful data collection. The system is now allowed to explore real dynamics, and no position restraints are used on the atoms.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd prod
# equilibrated structure from ../eq ; topology ../build ; index ../em ; mdp here in prod/
gmx grompp -f CG_prod_tut.mdp -c ../eq/npt.gro \
    -p ../build/topol.top -n ../em/index.ndx -o prod.tpr -maxwarn 1
gmx mdrun -deffnm prod -nb gpu -bonded gpu -v

# Post Processing

> **What happens here:** before looking at the trajectory we (a) build a second index file with extra groups for analysis, (b) fix the periodic boundaries so molecules are whole and centred, and (c) drop the water/ions to make files small and easy to view.

Before post processing, we have to create another index file. This helps in analysing different domains of the proteins, membranes, etc.

To make it easier to work with and to speed up the analysis, it is better to remove solvent (when we are not dealing with analysis related to water/ions). Solvent takes up a lot of space and can reduce the speed of post-processing.

For better organisation of files, we also create another folder, e.g., `analysis`.

In [ ]:
# The folder structure (build/ em/ eq/ prod/ analysis/) was already created
# in the setup cell at the top. Nothing to do here.
import os
print('Working dir:', os.getcwd())
print('Stage folders:', [d for d in ['build','em','eq','prod','analysis'] if os.path.isdir(d)])

## Creating index file

In this `index_analysis.ndx` file, we have different groups of atoms such as protein, membranes, transmembrane domain of the protein, etc. This helps to analyse the behaviour of different groups with each other.

> **What happens here:** we add a `Transmembrane` group (the membrane-spanning part of the protein, atoms 242-312) and a `not_SOL` group (everything except water and ions). `Transmembrane` is used to centre the view; `not_SOL` is what we keep in the cleaned trajectory. Built with `make_ndx` for the same reason as before (no MDAnalysis available).

In [ ]:
%%bash
# Build prod/index_analysis.ndx with extra groups for analysis.
# Numbers from 'gmx make_ndx -f prod.tpr': 1=Protein, 13=POPC, 14=W, 15=ION.
# Transmembrane = atoms 242-312 ; not_SOL = everything except water & ions.
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd prod
gmx make_ndx -f prod.tpr -o index_analysis.ndx << 'EOF'
name 14 MEM
1 | 13
name 17 GLYCOPROT
a 242-312
name 18 Transmembrane
! 15 & ! 16
name 19 not_SOL
q
EOF

## Correction of PBC

> **What happens here:** during the run, molecules that cross a box wall reappear on the opposite side, which can leave them looking 'broken' or scattered. `gmx trjconv -pbc mol` stitches each molecule back together and re-centres the system on the transmembrane domain, so the membrane looks like one continuous slab.

During MD simulation, periodic boundary conditions are used such that molecules leaving one side of the box re-enter from the opposite side. This creates an effectively infinite system, which is desirable.

However, for certain analysis or visualisation, this can result in “broken” molecules or fragmented trajectories.

To fix this, we apply a correction to make the molecules whole again. We use a GROMACS tool called `trjconv` for this.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd prod
# Centre on the transmembrane domain, make molecules whole, strip solvent.
gmx trjconv -f prod.xtc -s prod.tpr -o prod_noW.xtc -n index_analysis.ndx \
    -center -pbc mol -ur compact <<< "Transmembrane not_SOL"

## Visualising the result

> **What happens here:** we take a single clean frame from the production run and save it as a PDB file you can open in a molecular viewer. We use the **last** frame and make molecules whole, so the membrane shows up as a continuous slab instead of lipids scattered across the box edges.

In [ ]:
%%bash
# Clean single-frame structure for visualization (last frame, molecules whole).
module load GCC/14.2.0 GROMACS/2026.2-CUDA-12.8.0-SMP
cd prod
printf 'Transmembrane\nnot_SOL\n' | gmx trjconv \
    -f prod.xtc -s prod.tpr -n index_analysis.ndx \
    -o prod_vis.pdb -pbc mol -ur compact -center -dump 999999
echo '--- created: ---'
ls -la prod_vis.pdb

In [ ]:
import sys, os, subprocess, importlib

_t = os.path.expanduser('~/.local/py3dmol')
if _t not in sys.path:
    sys.path.insert(0, _t)

try:
    import py3Dmol
except ModuleNotFoundError:
    print('Installing py3Dmol into', _t, '(one-time)...')
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--target', _t, 'py3Dmol'],
        check=False
    )
    importlib.invalidate_caches()
    import py3Dmol

pdb_path = 'prod/prod_vis.pdb'
assert os.path.exists(pdb_path), f'{pdb_path} not found - run the snapshot cell above first'

with open(pdb_path) as fh:
    pdb_data = fh.read()

view = py3Dmol.view(width=900, height=650)
view.addModel(pdb_data, 'pdb')

# Default: all beads
view.setStyle(
    {},
    {'sphere': {'radius': 1.4, 'color': 'spectrum'}}
)

# Membrane POPC
view.setStyle(
    {'resn': 'POPC'},
    {'stick': {'color': 'silver', 'radius': 0.15, 'opacity': 0.35}}
)

# Glycan GYC
view.setStyle(
    {'resn': 'GYC'},
    {'sphere': {'radius': 1.9, 'color': 'red'}}
)

view.zoomTo()
view.setBackgroundColor('white')
view.show()